In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [2]:
data_log2 = pd.read_pickle('data_log2_Coimbra.pkl')

In [3]:
data_log2

Protein.Group,A0A075B6H7,A0A075B6K5,A0A075B6P5;P01615;A0A087WW87;P01614,A0A075B6S5;A0A0C4DH67;A0A0C4DH69,A0A0A0MRZ8;P04433,A0A0A0MS15,A0A0B4J1U7,A0A0B4J1X8,A0A0B4J1Y9,A0A0B4J2B5,...,Q9NRN5,Q9NYQ8,Q9P121,Q9P2S2,Q9UBP4,Q9UKI9;P09086;P14859,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7
109094,10.707256,5.411257,10.023893,7.638458,9.985671,5.338670,7.244240,9.311739,9.095131,10.772999,...,4.850699,4.290432,8.415112,6.759808,11.329084,9.030046,6.976387,7.458193,6.139852,6.967906
103208,10.510289,4.121894,9.891121,6.968736,9.974615,6.571228,6.063140,8.986320,8.741339,11.256226,...,4.335769,4.580694,8.203358,6.722548,11.352380,10.300444,6.761179,7.385923,6.398954,7.078834
106086,10.989870,5.323903,10.223121,7.522158,10.246075,6.168520,6.770049,9.322539,9.478424,11.403204,...,4.894512,4.759955,8.712379,6.741656,11.591747,9.908852,7.325251,7.835930,6.582637,7.505629
105634,11.706315,6.707000,11.341958,8.700062,11.423557,7.464734,5.915024,9.812152,9.418742,12.369186,...,5.383113,4.100952,8.368140,6.500428,11.666797,10.495495,6.355367,7.287186,6.226406,6.764420
106008,10.733066,4.704031,10.645001,8.005720,10.433419,5.200661,8.202614,9.718603,9.274483,11.513303,...,5.042412,4.073829,8.467292,6.067114,11.768011,10.377319,7.064355,7.343239,6.230420,7.659339
110203,10.245838,4.292936,9.241104,7.304219,10.118993,6.080417,8.769074,9.116656,8.588032,10.729825,...,4.639875,5.029488,8.565658,6.868131,11.799646,9.477698,7.183040,7.686171,6.173755,7.017710
108382,11.131838,5.668079,10.281976,7.698288,9.901271,6.452478,9.130254,9.069530,9.252261,11.197180,...,4.933927,3.923415,8.317399,6.656554,11.519843,10.041673,6.712032,7.335998,6.424015,7.993810
104968,10.706228,5.402255,10.087489,7.788953,9.729106,6.248379,8.236249,9.583534,8.581683,11.398770,...,5.188793,4.286970,8.294180,6.557016,11.373409,9.407812,6.957938,7.421526,6.570718,8.275324
107018,10.330312,5.685836,10.257482,7.405022,10.142733,6.457701,8.621645,9.630560,9.432364,11.696972,...,4.754866,3.918405,8.469768,6.735170,11.248604,10.364966,6.913189,7.510329,5.884200,6.478033
109456,9.491047,2.964088,8.829406,6.604319,9.283452,4.231041,4.319119,8.628099,7.791254,10.291482,...,4.915449,4.311648,8.384715,6.565285,11.583666,9.050121,7.160134,7.625658,6.226703,6.323552


In [4]:
import pickle

with open('list_groups_Coimbra_2.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD']


In [5]:
list_groups = pd.Series(list_groups)

In [6]:
'''
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold

# ======================
# DATA SETUP
# ======================
X = data_log2.copy()
y = np.array(list_groups)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# EXPERIMENT PARAMETERS
# ======================
k_values = [5, 10, 15, 20]
seeds = range(5)
all_k_results = []

print("Starting Sensitivity Analysis for K-features...")

# ======================
# OUTER LOOP: K-VALUES
# ======================
for k in k_values:
    k_seeds_mcc = []
    k_seeds_auc = []
    k_feature_union = set()

    print(f"\nTesting K = {k} proteins...")

    for seed in seeds:
        # 1. Split 70/30
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, stratify=y, random_state=seed
        )
        y_test_bin = (y_test == "MCI-AD").astype(int)

        # 2. Simple Feature Importance (5-Fold CV on Train only)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        fold_importances = []

        for train_idx, val_idx in cv.split(X_train, y_train):
            X_sub, y_sub = X_train.iloc[train_idx], y_train[train_idx]
            rf_fs = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=seed)
            rf_fs.fit(X_sub, y_sub)
            fold_importances.append(rf_fs.feature_importances_)

        # Average importance and Ranking
        mean_imp = np.mean(fold_importances, axis=0)
        ranking = pd.DataFrame({"protein": X.columns, "imp": mean_imp}).sort_values("imp", ascending=False)
        
        # Select Top-K
        top_features = ranking["protein"].iloc[:k].tolist()
        k_feature_union.update(top_features)

        # 3. Final Model Evaluation
        rf_final = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
        rf_final.fit(X_train[top_features], y_train)

        # Predictions
        y_pred = rf_final.predict(X_test[top_features])
        ad_idx = np.where(rf_final.classes_ == "MCI-AD")[0][0]
        y_prob = rf_final.predict_proba(X_test[top_features])[:, ad_idx]

        k_seeds_mcc.append(matthews_corrcoef(y_test, y_pred))
        k_seeds_auc.append(roc_auc_score(y_test_bin, y_prob))

    # Save aggregated results for this K
    res = {
        "K": k,
        "mean_mcc": np.mean(k_seeds_mcc),
        "std_mcc": np.std(k_seeds_mcc),
        "mean_auc": np.mean(k_seeds_auc),
        "std_auc": np.std(k_seeds_auc),
        "union_size": len(k_feature_union)
    }
    all_results.append(res)
    
    # INDIVIDUAL REPORT FOR EACH K
    print(f"--- REPORT FOR K={k} ---")
    print(f"Mean MCC: {res['mean_mcc']:.3f} (±{res['std_mcc']:.3f})")
    print(f"Mean AUC: {res['mean_auc']:.3f} (±{res['std_auc']:.3f})")
    print(f"Total Unique Proteins in Union: {res['union_size']}")

# ======================
# FINAL COMPARISON TABLE
# ======================
df_final = pd.DataFrame(all_results)
print("\n" + "="*50)
print("FINAL SENSITIVITY ANALYSIS SUMMARY")
print("="*50)
print(df_final[["K", "mean_mcc", "mean_auc", "union_size"]].to_string(index=False))
'''

'\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import matthews_corrcoef, roc_auc_score\nfrom sklearn.model_selection import train_test_split, StratifiedKFold\n\n# ======================\n# DATA SETUP\n# ======================\nX = data_log2.copy()\ny = np.array(list_groups)\ny_binary = (y == "MCI-AD").astype(int)\n\n# ======================\n# EXPERIMENT PARAMETERS\n# ======================\nk_values = [5, 10, 15, 20]\nseeds = range(5)\nall_k_results = []\n\nprint("Starting Sensitivity Analysis for K-features...")\n\n# ======================\n# OUTER LOOP: K-VALUES\n# ======================\nfor k in k_values:\n    k_seeds_mcc = []\n    k_seeds_auc = []\n    k_feature_union = set()\n\n    print(f"\nTesting K = {k} proteins...")\n\n    for seed in seeds:\n        # 1. Split 70/30\n        X_train, X_test, y_train, y_test = train_test_split(\n            X, y, test_size=0.3, stratify=y, random_state=seed\n        )\n 

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ======================
# DATI
# ======================
X = data_log2.copy()
y = np.array(list_groups)

# binarizza per AUC (importante!)
y_binary = (y == "MCI-AD").astype(int)

# ======================
# PARAMETRI GLOBALI
# ======================
thresholds = [ 0.7, 0.8, 0.9]
n_iterations = 20
seeds = range(15)

# ======================
# STORAGE
# ======================
all_results = []
feature_counter = {}
threshold_counter = {}

# ======================
# BOOTSTRAP
# ======================
def stratified_bootstrap(X, y):
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y[indices]


# ======================
# LOOP SEED
# ======================
union_top_features = set()

for seed in seeds:

    print(f"\n===== SEED {seed} =====")

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        stratify=y,
        random_state=seed
    )

    y_train_bin = (y_train == "MCI-AD").astype(int)
    y_test_bin = (y_test == "MCI-AD").astype(int)

    cv = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=seed)

    threshold_scores = []

    # ======================
    # CV per scegliere threshold
    # ======================
    for t in thresholds:

        fold_scores = []

        for train_idx, val_idx in cv.split(X_train, y_train):

            X_subtrain = X_train.iloc[train_idx]
            y_subtrain = y_train[train_idx]

            X_val = X_train.iloc[val_idx]
            y_val = y_train[val_idx]

            y_val_bin = (y_val == "MCI-AD").astype(int)

            # ===== bootstrap importance
            feature_names = X_subtrain.columns
            importance_matrix = np.zeros((n_iterations, len(feature_names)))

            for i in range(n_iterations):

                X_boot, y_boot = stratified_bootstrap(X_subtrain, y_subtrain)

                rf = RandomForestClassifier(
                    n_estimators=200,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=seed
                )

                rf.fit(X_boot, y_boot)
                importance_matrix[i] = rf.feature_importances_

            mean_importance = importance_matrix.mean(axis=0)

            ranking_df = pd.DataFrame({
                "protein": feature_names,
                "importance": mean_importance
            }).sort_values("importance", ascending=False)

            # normalizzazione
            ranking_df["importance"] /= ranking_df["importance"].sum()
            ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

            # selezione feature
            selected_features = ranking_df[
                ranking_df["cum_importance"] <= t
            ]["protein"]

            if len(selected_features) == 0:
                selected_features = ranking_df["protein"].iloc[:1]

            # train modello
            rf = RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=seed
            )

            rf.fit(X_subtrain[selected_features], y_subtrain)

            # valutazione
            y_pred = rf.predict(X_val[selected_features])
            mcc = matthews_corrcoef(y_val, y_pred)

            fold_scores.append(mcc)

        threshold_scores.append(np.mean(fold_scores))

    # ======================
    # best threshold
    # ======================
    best_threshold = thresholds[np.argmax(threshold_scores)]
    print("Best threshold:", best_threshold)

    # ======================
    # TRAIN FINALE (su tutto train)
    # ======================
    feature_names = X_train.columns
    importance_matrix = np.zeros((n_iterations, len(feature_names)))

    for i in range(n_iterations):

        X_boot, y_boot = stratified_bootstrap(X_train, y_train)

        rf = RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            n_jobs=-1,
            random_state=i
        )

        rf.fit(X_boot, y_boot)
        importance_matrix[i] = rf.feature_importances_

    mean_importance = importance_matrix.mean(axis=0)

    ranking_df = pd.DataFrame({
        "protein": feature_names,
        "importance": mean_importance
    }).sort_values("importance", ascending=False)

    ranking_df["importance"] /= ranking_df["importance"].sum()
    ranking_df["cum_importance"] = ranking_df["importance"].cumsum()

    top_features = ranking_df[
        ranking_df["cum_importance"] <= best_threshold
    ]["protein"]

    union_top_features.update(top_features)

    if len(top_features) == 0:
        top_features = ranking_df["protein"].iloc[:1]

    print("Numero feature:", len(top_features))

    # ======================
    # TEST
    # ======================
    rf_final = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42
    )

    rf_final.fit(X_train[top_features], y_train)

    y_pred = rf_final.predict(X_test[top_features])
    class_order = rf_final.classes_
    ad_index = np.where(class_order == "MCI-AD")[0][0]
    
    y_prob = rf_final.predict_proba(X_test[top_features])[:, ad_index]

    test_mcc = matthews_corrcoef(y_test, y_pred)
    test_auc = roc_auc_score(y_test_bin, y_prob)
    print("Class order:", rf_final.classes_)
    print("Test MCC:", test_mcc)
    print("Test AUC:", test_auc)

    # ======================
    # SALVATAGGIO
    # ======================
    all_results.append({
        "model": "RF",
        "seed": seed,
        "mcc": test_mcc,
        "auc": test_auc,
        "n_features": len(top_features),
        "best_threshold": best_threshold
    })

    # frequenza feature
    for f in top_features:
        feature_counter[f] = feature_counter.get(f, 0) + 1

    # frequenza threshold
    threshold_counter[best_threshold] = threshold_counter.get(best_threshold, 0) + 1


# ======================
# RISULTATI FINALI
# ======================
df_results = pd.DataFrame(all_results)

print("\n===== FINAL RESULTS =====")
print("Mean MCC:", df_results["mcc"].mean())
print("Mean AUC:", df_results["auc"].mean())

# frequenze
feature_freq = pd.Series(feature_counter).sort_values(ascending=False) / len(seeds)
threshold_freq = pd.Series(threshold_counter).sort_index() / len(seeds)

print("\nTop features (frequency):")
print(feature_freq.head(20))

print("\nThreshold frequency:")
print(threshold_freq)
print("\n===== UNIONE DI TUTTE LE TOP FEATURES =====")
print(f"Numero totale di proteine uniche trovate tra tutti i seed: {len(union_top_features)}")
print("Lista completa:")
print(sorted(list(union_top_features)))


===== SEED 0 =====
Best threshold: 0.9
Numero feature: 105
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8918825850158447
Test AUC: 1.0

===== SEED 1 =====
Best threshold: 0.9
Numero feature: 96
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.7774288420142416
Test AUC: 1.0

===== SEED 2 =====
Best threshold: 0.9
Numero feature: 94
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.7774288420142416
Test AUC: 1.0

===== SEED 3 =====
Best threshold: 0.9
Numero feature: 94
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.7774288420142416
Test AUC: 1.0

===== SEED 4 =====
Best threshold: 0.8
Numero feature: 60
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 1.0
Test AUC: 1.0

===== SEED 5 =====
Best threshold: 0.7
Numero feature: 44
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 1.0
Test AUC: 1.0

===== SEED 6 =====
Best threshold: 0.7
Numero feature: 46
Class order: ['MCI-AD' 'MCI-CT']
Test MCC: 0.8864052604279183
Test AUC: 1.0

===== SEED 7 =====
Best threshold: 0.8
Numero feature: 63
Class order: ['MCI-AD' 'M

In [8]:
len(feature_freq)
feature_freq.to_pickle('feature_rf_Coimbra.pkl')